In [32]:
import pandas as pd

df = pd.read_csv("edufeed_clean.csv")
df.head()

,department_name,year_since_first_review,star_rating,diff_index,tag_professor,num_student,post_date,student_star,student_difficult,help_useful,...,engagement_score,rating_difficulty_gap,sentiment_label,review_year,semester,semester_numeric,tag_count,help_ratio,review_length,is_low_quality
0,Astronomy department,11,4.7,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26.0,06/27/2017,5.0,3.0,0,...,0.25,2.7,positive,2017,Summer,2,3,0.0,252,0
1,Astronomy department,11,4.7,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26.0,04/16/2017,5.0,2.0,0,...,0.25,2.7,positive,2017,Spring,1,3,0.0,225,0
2,Astronomy department,11,4.7,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26.0,7/12/2016,4.0,3.0,0,...,0.25,2.7,positive,2016,Summer,2,3,0.0,190,0
3,Astronomy department,11,4.7,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26.0,8/12/2014,5.0,3.0,0,...,0.25,2.7,positive,2014,Summer,2,3,0.0,354,0
4,Astronomy department,11,4.7,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26.0,2/5/2014,5.0,1.0,0,...,0.25,2.7,positive,2014,Spring,1,3,0.0,338,0


In [34]:
df[['comments', 'sentiment_label']].head()

,comments,sentiment_label
0,"This class is hard, but its a two-in-one gen-e...",positive
1,Definitely going to choose Prof. Looney\'s cla...,positive
2,I overall enjoyed this class because the assig...,positive
3,"Yes, it\'s possible to get an A but you\'ll de...",positive
4,Professor Looney has great knowledge in Astron...,positive


In [35]:
df['sentiment_label'].value_counts()

,count
sentiment_label,
positive,11804
negative,5668
neutral,2505


In [36]:
!pip install vaderSentiment

In [37]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

In [38]:
def get_vader_sentiment(text):
    score = analyzer.polarity_scores(str(text))

    if score['compound'] >= 0.05:
        return "positive"
    elif score['compound'] <= -0.05:
        return "negative"
    else:
        return "neutral"

df['vader_prediction'] = df['comments'].apply(get_vader_sentiment)

In [39]:
df[['comments', 'sentiment_label', 'vader_prediction']].head()

,comments,sentiment_label,vader_prediction
0,"This class is hard, but its a two-in-one gen-e...",positive,positive
1,Definitely going to choose Prof. Looney\'s cla...,positive,positive
2,I overall enjoyed this class because the assig...,positive,positive
3,"Yes, it\'s possible to get an A but you\'ll de...",positive,positive
4,Professor Looney has great knowledge in Astron...,positive,positive


In [40]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(df['sentiment_label'], df['vader_prediction'])
print("VADER Accuracy:", accuracy)

VADER Accuracy: 0.7056114531711468


In [41]:
from sklearn.model_selection import train_test_split

X = df['comments']
y = df['sentiment_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

ValueError: np.nan is an invalid document, expected byte or unicode string.

In [43]:
df = df.dropna(subset=['comments'])

In [44]:
import spacy
import re

nlp = spacy.load("en_core_web_sm")

In [45]:
def anonymize_text(text):
    if pd.isnull(text):
        return text

    doc = nlp(text)
    new_text = text

    for ent in doc.ents:
        if ent.label_ == "PERSON":
            new_text = new_text.replace(ent.text, "[NAME]")

    new_text = re.sub(r'\S+@\S+', '[EMAIL]', new_text)
    new_text = re.sub(r'\b\d{10}\b', '[PHONE]', new_text)

    return new_text

In [46]:
df['anonymous_comment'] = df['comments'].apply(anonymize_text)

In [47]:
from sklearn.model_selection import train_test_split

X = df['anonymous_comment']
y = df['sentiment_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [50]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=200)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Logistic Regression Results:")
print(classification_report(y_test, y_pred))

Logistic Regression Results:
              precision    recall  f1-score   support

    negative       0.80      0.72      0.76      1209
     neutral       0.36      0.10      0.16       478
    positive       0.78      0.94      0.85      2308

    accuracy                           0.77      3995
   macro avg       0.64      0.59      0.59      3995
weighted avg       0.74      0.77      0.74      3995



In [51]:
y_pred = model.predict(X_test_tfidf)

In [52]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, y_pred)
print("Logistic Regression Accuracy:", acc)

Logistic Regression Accuracy: 0.7712140175219023


In [53]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[ 871   39  299]
 [ 125   48  305]
 [  98   48 2162]]
              precision    recall  f1-score   support

    negative       0.80      0.72      0.76      1209
     neutral       0.36      0.10      0.16       478
    positive       0.78      0.94      0.85      2308

    accuracy                           0.77      3995
   macro avg       0.64      0.59      0.59      3995
weighted avg       0.74      0.77      0.74      3995



In [54]:
df['sentiment_label'] = df['sentiment_label'].map({
    'positive': 1,
    'neutral': 0,
    'negative': -1
})

In [55]:
X = df[['student_difficult', 'review_length', 'sentiment_label']]
y = df['student_star']

In [56]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [57]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [58]:
from sklearn.metrics import mean_squared_error, r2_score

print("Linear Regression Results:")
print("MSE:", mean_squared_error(y_test, y_pred_lr))
print("R2 Score:", r2_score(y_test, y_pred_lr))

Linear Regression Results:
MSE: 0.19244168846265794
R2 Score: 0.9115720059759725


In [59]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\nRandom Forest Results:")
print("MSE:", mean_squared_error(y_test, y_pred_rf))
print("R2 Score:", r2_score(y_test, y_pred_rf))


Random Forest Results:
MSE: 0.2251390308215562
R2 Score: 0.8965474007679628


In [60]:
print("Baseline Model 1 Accuracy: ~77%")
print("Baseline Model 2 Linear Regression R2: ~0.91")
print("Baseline Model 2 Random Forest R2: ~0.89")

Baseline Model 1 Accuracy: ~77%
Baseline Model 2 Linear Regression R2: ~0.91
Baseline Model 2 Random Forest R2: ~0.89
